<a href="https://colab.research.google.com/github/saad823/CardioSync-RL/blob/main/Cardio_simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gymnasium

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

class SimulatedHeartRateEnv(gym.Env):
    def __init__(self):
      super(SimulatedHeartRateEnv, self).__init__()

      # Action space: 0 = Silence, 1 = Calming Music, 2 = Intense Rock Music
      self.action_space = spaces.Discrete(3)

      # State space: Simulated Heart Rate constrained between 40 and 180 BPM
      self.observation_space = spaces.Box(low=40.0, high=180.0, shape=(1,), dtype=np.float32)

      # Target healthy heart rate we want the AI to achieve
      self.target_bpm = 70.0

      # Track the current state internally
      self.current_bpm = 95.0 # The patient starts out stressed/anxious

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        # Reset the "patient" to a random stressed heart rate between 90 and 110 BPM
        self.current_bpm = float(self.np_random.uniform(90.0, 110.0))

        # Return state as a numpy array, plus an empty info dictionary
        state = np.array([self.current_bpm], dtype=np.float32)
        return state, {}

    def step(self, action):
        # 1. BIOLOGICAL DYNAMICS (The Math Model)
        # Add a tiny bit of random natural heart fluctuation (-1 to +1 BPM)
        natural_drift = self.np_random.uniform(-1.0, 1.0)
        self.current_bpm += natural_drift

        # Apply the impact of the AI's action
        if action == 0:
            # Silence: The anxious patient slowly drifts back up toward 100 BPM
            self.current_bpm += (100.0 - self.current_bpm) * 0.1
        elif action == 1:
            # Calming Music: Actively lowers the heart rate by a random factor
            reduction = self.np_random.uniform(2.0, 4.0)
            self.current_bpm -= reduction
        elif action == 2:
            # Intense Rock Music: Spikes the heart rate up
            spike = self.np_random.uniform(3.0, 6.0)
            self.current_bpm += spike

        # Keep the simulated heart rate within realistic boundaries
        self.current_bpm = np.clip(self.current_bpm, 40.0, 180.0)

        # 2. CALCULATE REWARD
        # The penalty is the absolute distance from our healthy target of 70 BPM.
        # Closer to 70 = reward closer to 0 (the maximum possible score).
        reward = -abs(self.current_bpm - self.target_bpm)

        # 3. TERMINATION CONDITIONS
        # End the simulation if the patient is "cured" (stabilized near 70)
        # or if they hit emergency high thresholds.
        terminated = bool(self.current_bpm <= 72.0 and self.current_bpm >= 68.0)

        # Safety cutoff: simulation fails if heart rate gets dangerously high
        if self.current_bpm > 140.0:
            terminated = True
            reward -= 100.0 # Heavy penalty for failing the patient

        truncated = False # Can be used for maximum step time limits

        state = np.array([self.current_bpm], dtype=np.float32)
        return state, reward, terminated, truncated, {}

In [ ]:
# Re-create the environment instance with the corrected class definition
env = SimulatedHeartRateEnv()
state, info = env.reset()

print(f"--- Session Started ---")
print(f"Starting Patient Heart Rate: {state[0]:.1f} BPM\n")

# Mapping number actions to readable text names
action_names = {0: "Silence", 1: "Play Calming Music", 2: "Play Rock Music"}

for step in range(15):
    # Pick a random action
    action = env.action_space.sample()

    # Push the action into the environment
    state, reward, terminated, truncated, _ = env.step(action)

    print(f"Step {step+1:02d} | Action: {action_names[action]:<20} | New HR: {state[0]:.1f} BPM | Score: {reward:.1f}")

    if terminated:
        if state[0] > 140:
            print("\n❌ CRITICAL CRASH: Patient heart rate spiked out of control! Safety override triggered.")
        else:
            print("\n✅ SUCCESS: Patient heart rate safely stabilized near 70 BPM!")
        break

--- Session Started ---
Starting Patient Heart Rate: 99.9 BPM

Step 01 | Action: Play Rock Music      | New HR: 105.1 BPM | Score: -35.1
Step 02 | Action: Play Rock Music      | New HR: 108.8 BPM | Score: -38.8
Step 03 | Action: Play Rock Music      | New HR: 113.3 BPM | Score: -43.3
Step 04 | Action: Play Rock Music      | New HR: 117.5 BPM | Score: -47.5
Step 05 | Action: Silence              | New HR: 115.9 BPM | Score: -45.9
Step 06 | Action: Play Calming Music   | New HR: 113.7 BPM | Score: -43.7
Step 07 | Action: Play Rock Music      | New HR: 117.1 BPM | Score: -47.1
Step 08 | Action: Play Calming Music   | New HR: 115.0 BPM | Score: -45.0
Step 09 | Action: Silence              | New HR: 113.5 BPM | Score: -43.5
Step 10 | Action: Silence              | New HR: 111.4 BPM | Score: -41.4
Step 11 | Action: Silence              | New HR: 109.5 BPM | Score: -39.5
Step 12 | Action: Play Calming Music   | New HR: 105.1 BPM | Score: -35.1
Step 13 | Action: Play Rock Music      | New HR: 

In [ ]:
!pip install stable-baselines3[extra]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 6.7 MB/s eta 0:00:00


### Training the PPO Agent
Now, let's import the PPO algorithm from Stable-Baselines3, instantiate our environment, and train the agent. We will also monitor the training using a basic evaluation.

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

# Create a vectorized wrapper for our custom environment
env_vec = make_vec_env(SimulatedHeartRateEnv, n_envs=1)

# Initialize PPO with a Multi-Input/MlpPolicy (standard feedforward neural network)
# We'll set a small learning rate and a custom tensorboard log directory if needed
model = PPO(
    "MlpPolicy",
    env_vec,
    verbose=1,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99
)

# Train the agent for 50,000 timesteps
print("Starting PPO Agent Training...")
model.learn(total_timesteps=50000)
print("Training Completed!")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Using cpu device
Starting PPO Agent Training...


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


----------------------------------
| rollout/           |           |
|    ep_len_mean     | 235       |
|    ep_rew_mean     | -1.01e+04 |
| time/              |           |
|    fps             | 1029      |
|    iterations      | 1         |
|    time_elapsed    | 1         |
|    total_timesteps | 2048      |
----------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 235         |
|    ep_rew_mean          | -1.01e+04   |
| time/                   |             |
|    fps                  | 763         |
|    iterations           | 2           |
|    time_elapsed         | 5           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.010875193 |
|    clip_fraction        | 0.0847      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.09       |
|    explained_variance   | -7.87e-05   |
|    learning_rate  

### Evaluating the Trained Agent
Let's test our newly trained PPO agent on a single episode to see if it successfully learns to stabilize the simulated heart rate around **70 BPM**.

In [ ]:
# Re-initialize a single test environment
test_env = SimulatedHeartRateEnv()
state, info = test_env.reset()

print(f"--- Trained PPO Agent Session Started ---")
print(f"Starting Patient Heart Rate: {state[0]:.1f} BPM\n")

action_names = {0: "Silence", 1: "Play Calming Music", 2: "Play Rock Music"}

for step in range(30):  # Let's give it up to 30 steps to stabilize
    # Use the trained model to predict the best action based on current state
    action, _states = model.predict(state, deterministic=True)

    # Take the step in the environment
    state, reward, terminated, truncated, _ = test_env.step(action)

    print(f"Step {step+1:02d} | Action Chosen: {action_names[int(action)]:<20} | New HR: {state[0]:.1f} BPM | Score: {reward:.1f}")

    if terminated:
        if state[0] > 140:
            print("\n❌ CRITICAL CRASH: Patient heart rate spiked out of control!")
        else:
            print(f"\n✅ SUCCESS: Patient heart rate stabilized perfectly at {state[0]:.1f} BPM!")
        break